In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import wavfile
from scipy import signal
from scipy.fft import fft
from scipy.signal import spectrogram
import io
import random
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, AdaBoostClassifier)
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

# Deep Learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, LSTM, GRU, Conv1D, MaxPooling1D,
                                      Flatten, Dropout, BatchNormalization,
                                      Bidirectional, GlobalAveragePooling1D)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("=" * 80)
print("DEEPFAKE VOICE DETECTION - COMPLETE PIPELINE WITH ENHANCED EDA")
print("=" * 80)

# --- 1. DATA LOADING ---
print("\n[STEP 1] Loading WaveFake Dataset from Hugging Face...")
try:
    df = pd.read_parquet("hf://datasets/ajaykarthick/wavefake-audio/data/partition0-00000-of-00001.parquet")
    print(f"✓ Loaded {len(df)} samples successfully")
except Exception as e:
    print(f"✗ Error loading data: {e}")
    print("Creating mock data for demonstration...")

    def create_mock_audio_bytes(duration_seconds=3, sample_rate=16000):
        num_samples = int(duration_seconds * sample_rate)
        audio_data = (np.random.normal(0, 1, num_samples) * 32767).astype(np.int16)
        buffer = io.BytesIO()
        wavfile.write(buffer, rate=sample_rate, data=audio_data)
        return {'bytes': buffer.getvalue()}

    unique_labels = ['R', 'WF1', 'WF2', 'WF3', 'WF4', 'WF5', 'WF6', 'WF7']
    N_SAMPLES = 800
    data = {
        'audio_id': [f'LJ001-0001{label}{i}' for label in unique_labels for i in range(N_SAMPLES // 8)],
        'real_or_fake': [label for label in unique_labels for _ in range(N_SAMPLES // 8)],
    }
    df = pd.DataFrame(data)
    df['audio'] = df.apply(lambda row: create_mock_audio_bytes(duration_seconds=random.uniform(2.5, 4.5)), axis=1)
    print(f"✓ Created {len(df)} mock samples")

# --- 2. BINARY LABEL CONVERSION ---
print("\n[STEP 2] Converting Labels to Binary Format...")
df['label'] = df['real_or_fake'].apply(lambda x: 0 if x == 'R' else 1)
print(f"✓ Binary conversion complete:")
print(f"  - Real (0): {(df['label'] == 0).sum()} samples")
print(f"  - Fake (1): {(df['label'] == 1).sum()} samples")
print(f"  - Class imbalance ratio: {(df['label'] == 1).sum() / (df['label'] == 0).sum():.2f}")

# --- 3. AUDIO PROCESSING FUNCTIONS ---
def get_audio_info(audio_bytes_dict):
    """Extract audio data from bytes"""
    try:
        buffer = io.BytesIO(audio_bytes_dict['bytes'])
        sample_rate, data = wavfile.read(buffer)
        if data.ndim > 1:
            data = data[:, 0]  # Convert to mono
        return sample_rate, data
    except:
        return 0, np.array([])

def extract_advanced_features(audio_bytes_dict, target_length=16000):
    """Extract comprehensive audio features for ML models"""
    sample_rate, data = get_audio_info(audio_bytes_dict)

    if sample_rate == 0 or len(data) == 0:
        return None

    # Normalize audio
    data = data.astype(np.float32)
    if np.max(np.abs(data)) > 0:
        data = data / np.max(np.abs(data))

    # Pad or truncate to target length
    if len(data) < target_length:
        data = np.pad(data, (0, target_length - len(data)), mode='constant')
    else:
        data = data[:target_length]

    features = {}

    # 1. Time Domain Features
    features['mean'] = np.mean(data)
    features['std'] = np.std(data)
    features['max'] = np.max(data)
    features['min'] = np.min(data)
    features['median'] = np.median(data)
    features['rms'] = np.sqrt(np.mean(data**2))
    features['zero_crossing_rate'] = np.sum(np.abs(np.diff(np.sign(data)))) / (2 * len(data))

    # 2. Frequency Domain Features (FFT)
    fft_vals = np.abs(fft(data))
    fft_vals = fft_vals[:len(fft_vals)//2]  # Use only positive frequencies

    features['spectral_centroid'] = np.sum(fft_vals * np.arange(len(fft_vals))) / np.sum(fft_vals) if np.sum(fft_vals) > 0 else 0
    features['spectral_rolloff'] = np.percentile(fft_vals, 85)
    features['spectral_bandwidth'] = np.std(fft_vals)
    features['spectral_contrast'] = np.max(fft_vals) - np.min(fft_vals)

    # 3. Power Spectral Density Features
    try:
        f, Pxx = signal.welch(data, fs=sample_rate, nperseg=min(1024, len(data)//2))
        features['psd_mean'] = np.mean(Pxx)
        features['psd_std'] = np.std(Pxx)
        features['psd_max'] = np.max(Pxx)

        # Low, mid, high frequency power
        freq_bands = [(0, 500), (500, 2000), (2000, 4000), (4000, 8000)]
        for i, (low, high) in enumerate(freq_bands):
            mask = (f >= low) & (f < high)
            features[f'power_band_{i}'] = np.sum(Pxx[mask])
    except:
        pass

    # 4. MFCC-like features (simplified)
    try:
        # Mel-scale approximation
        f_mel = 2595 * np.log10(1 + f / 700)
        for i in range(5):  # 5 mel bands
            band_mask = (f_mel >= i*500) & (f_mel < (i+1)*500)
            if np.any(band_mask):
                features[f'mel_band_{i}'] = np.mean(Pxx[band_mask])
    except:
        pass

    # 5. Entropy
    if len(data) > 0:
        hist, _ = np.histogram(data, bins=50)
        hist = hist / np.sum(hist)
        hist = hist[hist > 0]
        features['entropy'] = -np.sum(hist * np.log2(hist))

    return features

def augment_audio(data, sample_rate, augmentation_type='noise'):
    """Apply various augmentation techniques"""
    data = data.astype(np.float32)

    if augmentation_type == 'noise':
        # Add Gaussian noise
        noise = np.random.normal(0, 0.005, len(data))
        return data + noise

    elif augmentation_type == 'pitch_shift':
        # Simple pitch shifting via resampling
        factor = np.random.uniform(0.9, 1.1)
        indices = np.arange(0, len(data), factor)
        indices = indices[indices < len(data)].astype(int)
        return data[indices]

    elif augmentation_type == 'time_stretch':
        # Time stretching
        factor = np.random.uniform(0.9, 1.1)
        indices = np.round(np.linspace(0, len(data)-1, int(len(data)*factor))).astype(int)
        return data[indices]

    elif augmentation_type == 'volume':
        # Volume adjustment
        factor = np.random.uniform(0.7, 1.3)
        return data * factor

    elif augmentation_type == 'shift':
        # Time shift
        shift = np.random.randint(-len(data)//10, len(data)//10)
        return np.roll(data, shift)

    return data

# --- 4. FEATURE EXTRACTION ---
print("\n[STEP 3] Extracting Advanced Features from Audio...")
feature_list = []
labels_list = []

for idx, row in df.iterrows():
    features = extract_advanced_features(row['audio'])
    if features is not None:
        feature_list.append(features)
        labels_list.append(row['label'])

    if (idx + 1) % 100 == 0:
        print(f"  Processed {idx + 1}/{len(df)} samples...", end='\r')

print(f"\n✓ Extracted features from {len(feature_list)} samples")

# Create feature DataFrame
X_features = pd.DataFrame(feature_list)
y_labels = np.array(labels_list)

print(f"✓ Feature matrix shape: {X_features.shape}")
print(f"✓ Feature columns: {list(X_features.columns)}")

# ============================================================================
# COMPREHENSIVE DATASET ANALYSIS
# ============================================================================
print("\n" + "=" * 80)
print("COMPREHENSIVE DATASET ANALYSIS")
print("=" * 80)

# --- ANALYSIS 1: BINARY CLASS FREQUENCY ---
print("\n[ANALYSIS 1] Binary Class Distribution")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart
class_counts = pd.Series(y_labels).value_counts().sort_index()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Real (0)', 'Fake (1)'], class_counts.values, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Class Distribution (Bar Chart)', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['Real', 'Fake'], autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Class Distribution (Pie Chart)', fontsize=14, fontweight='bold')

# Statistics table
stats_text = f"""
Class Distribution Statistics:

Real Samples:     {class_counts[0]}
Fake Samples:     {class_counts[1]}
Total Samples:    {len(y_labels)}

Proportion Real:  {class_counts[0]/len(y_labels)*100:.2f}%
Proportion Fake:  {class_counts[1]/len(y_labels)*100:.2f}%

Imbalance Ratio:  {class_counts[1]/class_counts[0]:.3f}
"""
axes[2].text(0.1, 0.5, stats_text, fontsize=11, verticalalignment='center',
             fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
axes[2].axis('off')
axes[2].set_title('Statistics Summary', fontsize=14, fontweight='bold')

plt.suptitle('Binary Class Frequency Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"✓ Class distribution visualized")
print(f"  Real: {class_counts[0]} | Fake: {class_counts[1]}")

# --- ANALYSIS 2: PAIR PLOT ---
print("\n[ANALYSIS 2] Pair Plot Analysis")

# Select key features for pair plot (too many features will be cluttered)
key_features = ['rms', 'zero_crossing_rate', 'spectral_centroid',
                'spectral_bandwidth', 'psd_mean', 'entropy']

# Create DataFrame with selected features and labels
pair_plot_data = X_features[key_features].copy()
pair_plot_data['Class'] = ['Real' if y == 0 else 'Fake' for y in y_labels]

# Create pair plot
pair_plot = sns.pairplot(pair_plot_data, hue='Class', palette={'Real': '#2ecc71', 'Fake': '#e74c3c'},
                         diag_kind='kde', plot_kws={'alpha': 0.6, 's': 20},
                         diag_kws={'alpha': 0.7, 'linewidth': 2})
pair_plot.fig.suptitle('Pair Plot - Key Features by Class', fontsize=16, fontweight='bold', y=1.01)
plt.show()

print(f"✓ Pair plot generated for {len(key_features)} key features")

# --- ANALYSIS 3: ADVANCED CORRELATION ANALYSIS ---
print("\n[ANALYSIS 3] Advanced Correlation Analysis")

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Correlation heatmap
correlation_matrix = X_features.corr()
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            ax=axes[0, 0], vmin=-1, vmax=1)
axes[0, 0].set_title('Feature Correlation Heatmap (All Features)', fontsize=14, fontweight='bold')

# Top correlations
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_pairs.append({
            'Feature 1': correlation_matrix.columns[i],
            'Feature 2': correlation_matrix.columns[j],
            'Correlation': correlation_matrix.iloc[i, j]
        })
corr_df = pd.DataFrame(corr_pairs).sort_values('Correlation', key=abs, ascending=False).head(15)

y_pos = np.arange(len(corr_df))
axes[0, 1].barh(y_pos, corr_df['Correlation'].values,
                color=['red' if x < 0 else 'blue' for x in corr_df['Correlation'].values],
                alpha=0.7, edgecolor='black')
axes[0, 1].set_yticks(y_pos)
axes[0, 1].set_yticklabels([f"{row['Feature 1'][:10]}...{row['Feature 2'][:10]}"
                             for _, row in corr_df.iterrows()], fontsize=8)
axes[0, 1].set_xlabel('Correlation Coefficient', fontsize=11)
axes[0, 1].set_title('Top 15 Feature Correlations', fontsize=14, fontweight='bold')
axes[0, 1].axvline(x=0, color='black', linestyle='--', linewidth=0.8)
axes[0, 1].grid(axis='x', alpha=0.3)

# Correlation with target
X_with_target = X_features.copy()
X_with_target['target'] = y_labels
target_corr = X_with_target.corr()['target'].drop('target').sort_values(key=abs, ascending=False)

axes[1, 0].barh(range(len(target_corr[:15])), target_corr[:15].values,
                color=['green' if x > 0 else 'red' for x in target_corr[:15].values],
                alpha=0.7, edgecolor='black')
axes[1, 0].set_yticks(range(len(target_corr[:15])))
axes[1, 0].set_yticklabels(target_corr[:15].index, fontsize=9)
axes[1, 0].set_xlabel('Correlation with Target', fontsize=11)
axes[1, 0].set_title('Top 15 Features Correlated with Target', fontsize=14, fontweight='bold')
axes[1, 0].axvline(x=0, color='black', linestyle='--', linewidth=0.8)
axes[1, 0].grid(axis='x', alpha=0.3)

# Feature variance
feature_var = X_features.var().sort_values(ascending=False)[:15]
axes[1, 1].barh(range(len(feature_var)), feature_var.values, color='purple', alpha=0.7, edgecolor='black')
axes[1, 1].set_yticks(range(len(feature_var)))
axes[1, 1].set_yticklabels(feature_var.index, fontsize=9)
axes[1, 1].set_xlabel('Variance', fontsize=11)
axes[1, 1].set_title('Top 15 Features by Variance', fontsize=14, fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.suptitle('Comprehensive Correlation Analysis', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"✓ Correlation analysis complete")
print(f"  Highest correlation with target: {target_corr.iloc[0]:.4f} ({target_corr.index[0]})")

# --- ANALYSIS 4: SPECTROGRAM ANALYSIS ---
print("\n[ANALYSIS 4] Spectrogram Analysis")

# Select 2 real and 2 fake samples for spectrogram visualization
real_indices = np.where(y_labels == 0)[0][:2]
fake_indices = np.where(y_labels == 1)[0][:2]
sample_indices = list(real_indices) + list(fake_indices)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, sample_idx in enumerate(sample_indices):
    sample_label = 'Real' if y_labels[sample_idx] == 0 else 'Fake'
    audio_data = df.iloc[sample_idx]['audio']
    sample_rate, data = get_audio_info(audio_data)

    if sample_rate > 0 and len(data) > 0:
        # Normalize
        data = data.astype(np.float32)
        if np.max(np.abs(data)) > 0:
            data = data / np.max(np.abs(data))

        # Compute spectrogram
        f, t, Sxx = spectrogram(data, fs=sample_rate, nperseg=256)

        # Plot
        im = axes[idx].pcolormesh(t, f, 10 * np.log10(Sxx + 1e-10), shading='gouraud', cmap='viridis')
        axes[idx].set_ylabel('Frequency [Hz]', fontsize=11)
        axes[idx].set_xlabel('Time [sec]', fontsize=11)
        axes[idx].set_title(f'{sample_label} Audio - Sample {idx+1}', fontsize=12, fontweight='bold')
        axes[idx].set_ylim([0, 8000])  # Focus on speech frequencies
        plt.colorbar(im, ax=axes[idx], label='Power [dB]')

plt.suptitle('Spectrogram Analysis: Real vs Fake Audio', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"✓ Spectrogram analysis complete for {len(sample_indices)} samples")

# --- ANALYSIS 5: STATISTICAL DISTRIBUTION COMPARISON ---
print("\n[ANALYSIS 5] Statistical Distribution Comparison")

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

# Select top features by correlation with target
top_features = target_corr[:9].index.tolist()

for idx, feature in enumerate(top_features):
    # Box plot
    real_data = X_features[y_labels == 0][feature]
    fake_data = X_features[y_labels == 1][feature]

    bp = axes[idx].boxplot([real_data, fake_data], labels=['Real', 'Fake'],
                           patch_artist=True, widths=0.6,
                           boxprops=dict(facecolor='lightblue', alpha=0.7),
                           medianprops=dict(color='red', linewidth=2),
                           whiskerprops=dict(linewidth=1.5),
                           capprops=dict(linewidth=1.5))

    # Color boxes
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')

    axes[idx].set_title(f'{feature}', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Value', fontsize=10)
    axes[idx].grid(axis='y', alpha=0.3)

    # Add mean markers
    axes[idx].plot([1, 2], [real_data.mean(), fake_data.mean()],
                   'D', color='black', markersize=8, label='Mean')
    axes[idx].legend(fontsize=8)

plt.suptitle('Box Plot Comparison - Top Features by Target Correlation',
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"✓ Box plot comparison complete for top {len(top_features)} features")

# --- ANALYSIS 6: VIOLIN PLOTS ---
print("\n[ANALYSIS 6] Violin Plot Analysis")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feature in enumerate(key_features):
    violin_data = pd.DataFrame({
        'Value': X_features[feature],
        'Class': ['Real' if y == 0 else 'Fake' for y in y_labels]
    })

    sns.violinplot(data=violin_data, x='Class', y='Value', ax=axes[idx],
                   palette={'Real': '#2ecc71', 'Fake': '#e74c3c'},
                   inner='quartile', alpha=0.7)
    axes[idx].set_title(f'{feature}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('', fontsize=10)
    axes[idx].set_ylabel('Value', fontsize=10)
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Violin Plot Analysis - Distribution Shape Comparison',
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"✓ Violin plot analysis complete")

# --- ANALYSIS 7: FEATURE IMPORTANCE (QUICK RANDOM FOREST) ---
print("\n[ANALYSIS 7] Feature Importance Analysis")

# Quick Random Forest to get feature importance
rf_quick = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf_quick.fit(X_features, y_labels)
feature_importance = pd.DataFrame({
    'Feature': X_features.columns,
    'Importance': rf_quick.feature_importances_
}).sort_values('Importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
axes[0].barh(range(15), feature_importance['Importance'][:15].values,
             color='teal', alpha=0.7, edgecolor='black')
axes[0].set_yticks(range(15))
axes[0].set_yticklabels(feature_importance['Feature'][:15], fontsize=10)
axes[0].set_xlabel('Importance Score', fontsize=11)
axes[0].set_title('Top 15 Features by Random Forest Importance', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Cumulative importance
cumsum = np.cumsum(feature_importance['Importance'].values)
axes[1].plot(range(len(cumsum)), cumsum, marker='o', color='darkblue', linewidth=2, markersize=4)
axes[1].axhline(y=0.95, color='red', linestyle='--', linewidth=2, label='95% Threshold')
axes[1].fill_between(range(len(cumsum)), cumsum, alpha=0.3)
axes[1].set_xlabel('Number of Features', fontsize=11)
axes[1].set_ylabel('Cumulative Importance', fontsize=11)
axes[1].set_title('Cumulative Feature Importance', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=10)

plt.suptitle('Feature Importance Analysis', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"✓ Feature importance analysis complete")
print(f"  Top feature: {feature_importance.iloc[0]['Feature']} (Importance: {feature_importance.iloc[0]['Importance']:.4f})")

print("\n" + "=" * 80)
print("DATASET ANALYSIS COMPLETE!")
print("=" * 80)

# --- 5. DATA AUGMENTATION TO BALANCE CLASSES ---
print("\n[STEP 4] Applying Advanced Augmentation to Balance Dataset...")

# Count current samples per class
real_count = np.sum(y_labels == 0)
fake_count = np.sum(y_labels == 1)
target_per_class = 1500

print(f"  Current distribution - Real: {real_count}, Fake: {fake_count}")
print(f"  Target: {target_per_class} samples per class")

augmented_features = []
augmented_labels = []

# Process each class
for label_val in [0, 1]:
    label_name = "Real" if label_val == 0 else "Fake"
    current_count = np.sum(y_labels == label_val)
    needed = target_per_class - current_count

    print(f"\n  Processing {label_name} class:")
    print(f"    Current: {current_count}, Need: {needed} more samples")

    if needed > 0:
        # Get indices of this class
        class_indices = np.where(y_labels == label_val)[0]
        class_df_indices = df[df['label'] == label_val].index.tolist()

        # Select samples to augment
        augment_count = 0
        augmentation_types = ['noise', 'pitch_shift', 'time_stretch', 'volume', 'shift']

        while augment_count < needed and len(class_df_indices) > 0:
            # Randomly select a sample
            idx = np.random.choice(class_df_indices)
            audio_data = df.loc[idx, 'audio']

            # Apply random augmentation
            aug_type = np.random.choice(augmentation_types)
            sample_rate, data = get_audio_info(audio_data)

            if sample_rate > 0 and len(data) > 0:
                # Apply augmentation
                augmented_data = augment_audio(data, sample_rate, aug_type)

                # Convert back to bytes format
                augmented_data_int = (augmented_data * 32767).astype(np.int16)
                buffer = io.BytesIO()
                wavfile.write(buffer, rate=sample_rate, data=augmented_data_int)
                augmented_audio = {'bytes': buffer.getvalue()}

                # Extract features
                features = extract_advanced_features(augmented_audio)
                if features is not None:
                    augmented_features.append(features)
                    augmented_labels.append(label_val)
                    augment_count += 1

                    if augment_count % 100 == 0:
                        print(f"    Generated {augment_count}/{needed} samples...", end='\r')

print(f"\n✓ Generated {len(augmented_features)} augmented samples")

# Combine original and augmented data
X_augmented = pd.DataFrame(augmented_features)
X_combined = pd.concat([X_features, X_augmented], ignore_index=True)
y_combined = np.concatenate([y_labels, augmented_labels])

print(f"✓ Final dataset shape: {X_combined.shape}")
print(f"  - Real (0): {np.sum(y_combined == 0)} samples")
print(f"  - Fake (1): {np.sum(y_combined == 1)} samples")

# Handle missing values
X_combined = X_combined.fillna(X_combined.mean())

# Feature Scaling
print("\n[STEP 5] Applying Feature Scaling...")
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_combined)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_combined, test_size=0.2, random_state=42, stratify=y_combined
)

print(f"✓ Training set: {X_train.shape[0]} samples")
print(f"✓ Test set: {X_test.shape[0]} samples")

print("\n✓ Dataset preparation complete - Ready for model training!")
print("=" * 80)

Output hidden; open in https://colab.research.google.com to view.